In [11]:
import requests
import pandas as pd 
url = 'http://thpt-lequydon.edu.vn/Portals/1/thayca/HANNOM/sach/Tailieuhannomquocngu/nhidomai.htm'

response = requests.get(url)
response.encoding = response.apparent_encoding
content = response.text

with open('raw_data.txt', 'w', encoding="utf-8") as file:
    file.write(content)
    

In [8]:
import pandas as pd
from bs4 import BeautifulSoup

# 1. Đọc file Excel đã lưu từ bước trước
input_file = 'web.xlsx'
df = pd.read_excel(input_file)

def clean_html(html_content):
    if pd.isna(html_content):
        return ""
    
    # 2. Sử dụng BeautifulSoup để bóc tách HTML
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Loại bỏ các thẻ không mong muốn như script hoặc style (nếu có)
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

    # 3. Lấy văn bản thuần
    # separator='\n' giúp giữ lại xuống dòng giữa các đoạn văn
    text = soup.get_text(separator='\n')
    
    # Làm sạch khoảng trắng thừa
    lines = (line.strip() for line in text.splitlines())
    clean_text = '\n'.join(chunk for chunk in lines if chunk)
    
    return clean_text

# 4. Áp dụng hàm xử lý cho toàn bộ cột CONTENT
df['CLEAN_TEXT'] = df['CONTENT'].apply(clean_html)

# 5. Lưu lại kết quả vào file Excel mới
output_file = 'web_cleaned.xlsx'
df.to_excel(output_file, index=False)

print(f"Xử lý xong! Nội dung sạch đã được lưu tại: {output_file}")

Xử lý xong! Nội dung sạch đã được lưu tại: web_cleaned.xlsx


In [18]:
import pandas as pd
from bs4 import BeautifulSoup


def read_file_contents(file_path):
    try:
        # Open the file in read mode ('r') with UTF-8 encoding
        with open(file_path, 'r', encoding='utf-8') as file:
            # Read the entire file content
            html_content = file.read()
            return html_content
    except FileNotFoundError:
        return f"Error: The file '{file_path}' was not found."
    except PermissionError:
        return f"Error: Permission denied for '{file_path}'."
    except Exception as e:
        return f"An unexpected error occurred: {e}"
    
# 1. Đọc file Excel hiện có
html_content = read_file_contents('raw_data.txt')
# Lấy nội dung HTML từ cột CONTENT (dòng đầu tiên)
# html_content = df['CONTENT'].iloc[0]

# 2. Xử lý bằng BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

# Loại bỏ các thẻ rác (script, style)
for element in soup(["script", "style", "meta", "title"]):
    element.decompose()

# Lấy plain text với phân tách dòng rõ ràng
text = soup.get_text(separator='\n')

# 3. Làm sạch khoảng trắng và dòng trống
lines = (line.strip() for line in text.splitlines())
clean_text = '\n'.join(chunk for chunk in lines if chunk)

# 4. Ghi nội dung ra file .txt
with open('clean.txt', 'w', encoding='utf-8') as f:
    f.write(clean_text)

print("Đã trích xuất xong! Kiểm tra file 'clean.txt'.")

Đã trích xuất xong! Kiểm tra file 'clean.txt'.


In [17]:
import re
import html

def clean_html_v2(html_str):
    # 1. Loại bỏ các dấu xuống dòng, tab rác trong mã nguồn (HTML vốn coi chúng là khoảng trắng)
    # Bước này cực kỳ quan trọng để "Nhà Đường... \n Đức Tông" thành "Nhà Đường... Đức Tông"
    s = html_str.replace('\r', '').replace('\n', ' ').replace('\t', ' ')
    
    # 2. Giải mã các ký tự đặc biệt (ví dụ: &#272; thành Đ)
    s = html.unescape(s)
    
    # 3. Tạo dấu ngắt dòng chủ động tại các thẻ xuống dòng thật sự của HTML
    # Thay thế <br>, <p>, <div>, <tr> bằng một ký hiệu đặc biệt [[NL]]
    s = re.sub(r'<(br|BR)\s*/?>', '[[NL]]', s)
    s = re.sub(r'</?(p|P|div|DIV|tr|TR|h\d|H\d|li|LI)[^>]*>', '[[NL]]', s)
    
    # 4. Loại bỏ tất cả các thẻ HTML còn lại
    s = re.sub(r'<[^>]+>', '', s)
    
    # 5. Tách văn bản theo ký hiệu [[NL]] đã tạo
    segments = s.split('[[NL]]')
    
    cleaned_segments = []
    for seg in segments:
        # Xử lý khoảng trắng thừa bên trong mỗi đoạn
        clean_seg = " ".join(seg.split()).strip()
        if clean_seg:
            cleaned_segments.append(clean_seg)
            
    # 6. Nối lại bằng dấu xuống dòng thật sự
    return "\n".join(cleaned_segments)

# --- THỰC THI ---
# Đọc file raw_data.txt của bạn
with open('raw_data.txt', 'r', encoding='utf-8') as f:
    full_content = f.read()

# Xử lý làm sạch
result = clean_html_v2(full_content)

# Ghi ra file kết quả
with open('noidung_nhidomai_chuan.txt', 'w', encoding='utf-8') as f:
    f.write(result)

print("Đã xử lý xong! Các câu thơ 'Đức Tông' đã được nối liền mạch.")

Đã xử lý xong! Các câu thơ 'Đức Tông' đã được nối liền mạch.
